In [1]:
from ultralytics import YOLO

# Cargar tu modelo fine-tuned (ejemplo: best.pt)
# model = YOLO("/home/gonzadzz/GitHub/yolo11_container/YOLO_IDs/ID_YOLO_container/weights/best.pt")
model = YOLO("/home/gnz/GitHub/yolo11_container/YOLO_IDs/ID_YOLO_container/weights/best.pt")

# Ejecutar sobre un video (puede ser archivo local o stream RTSP/HTTP)
results = model.predict(
    source="/home/gnz/GitHub/yolo11_container/videos/short_youtube.mp4",   # también soporta rtsp://, http:// o 0 (cámara web)
    conf=0.5,             # confianza mínima
    save=True,            # guarda video con bounding boxes
    show=True             # muestra en pantalla (requiere cv2)
)



WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/321) /home/gnz/GitHub/yolo11_container/videos/short_youtube.mp4: 640x384 (no detections), 204.6ms
video 1/1 (frame 2/321) /home/gnz/GitHub/yolo11_container/videos/short_youtube.mp4: 640x384 (no detections), 112.4ms
video 1/1 (frame 3/321) /home/gnz/GitHub/yolo11_container/videos/short_youtube.mp4: 640x384 (no detections), 100.6ms
video 1/1 (frame 4/321) /home/gnz/GitHub/yolo11_container/videos/short_youtube.mp4: 640x384 (no detection

In [4]:
from ultralytics import YOLO
from PIL import Image
import numpy as np
import easyocr
import re
import cv2


# Cargar modelos
ids_model = YOLO("/home/gnz/GitHub/yolo11_container/YOLO_IDs/ID_YOLO_container/weights/best.pt")
# ids_model = YOLO("/home/gonzadzz/GitHub/yolo11_container/YOLO_IDs/ID_YOLO_container/weights/best.pt")
char_model = YOLO("/home/gnz/GitHub/yolo11_container/YOLO_Characters/Character_YOLO_container_finetune_large/weights/best.pt")
# char_model = YOLO("/home/gonzadzz/GitHub/yolo11_container/YOLO_Characters/Character_YOLO_container_finetune_extra_large_phase2/weights/best.pt")
# Inicializar EasyOCR
ocr_model = easyocr.Reader(['en','es'])

# Reglas RegEx para validación
rules = {
    "code-container": {"attribute": "code-container", "regex": r"^[A-Z]{4}\d{7}$"},
    "cn-11": {"attribute": "cn-11", "regex": r"^[A-Z]{4}\d{7}$"},
    "cn-4": {"attribute": "cn-4", "regex": r"^[A-Z]{4}$"},
    "cn-7": {"attribute": "cn-7", "regex": r"^\d{7}$"},
    "iso-type": {"attribute": "iso-type", "regex": r"^.{2}[A-Z0-9]{2}$"}  # ajustado a ISO tipo
}

# Reglas de validación
def parse_detecciones(detecciones, rules):
    parsed = {}
    for key, value in detecciones.items():
        if key in rules:
            attr = rules[key]["attribute"]
            pattern = rules[key]["regex"]

            # Validar con regex
            match = bool(re.match(pattern, value))

            # Resultado estructurado para Gradio JSON
            parsed[attr] = {
                "value": value,
                "valid": "✔️" if match else "❌"
            }
    return parsed

def calculate_check_digit(container_code: str) -> str | None:
    """
    Calcula o valida el dígito de check digit de un código de contenedor ISO 6346.
    
    - Si el código tiene 10 caracteres → calcula el dígito y devuelve el código completo (11).
    - Si el código tiene 11 caracteres → valida el dígito, si es correcto devuelve el mismo,
      si es incorrecto devuelve el código corregido.
    - Si el código tiene más de 11 → toma los primeros 10, calcula el dígito y devuelve esos 11.
    - Si los primeros 4 caracteres no son letras, devuelve None.
    """
    # Validar longitud mínima
    if len(container_code) < 10:
        return None

    # Validar que los primeros 4 sean letras
    if not container_code[:4].isalpha():
        return None

    # Tomar primeros 10 caracteres
    code_10 = container_code[:10]

    # Mapeo de letras a valores ISO 6346
    letter_values = {
        'A': 10, 'B': 12, 'C': 13, 'D': 14, 'E': 15, 'F': 16, 'G': 17, 'H': 18, 'I': 19, 'J': 20,
        'K': 21, 'L': 23, 'M': 24, 'N': 25, 'O': 26, 'P': 27, 'Q': 28, 'R': 29, 'S': 30, 'T': 31,
        'U': 32, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38
    }

    # Convertir a valores numéricos
    values = []
    for char in code_10:
        if char.isalpha():
            values.append(letter_values[char.upper()])
        else:
            values.append(int(char))

    # Calcular suma ponderada con 2^(posición)
    total = sum(val * (2 ** i) for i, val in enumerate(values))

    # Resto módulo 11
    check_digit = total % 11
    if check_digit == 10:
        check_digit = 0

    # Caso 10 caracteres → devolver con check digit
    if len(container_code) == 10:
        return code_10 + str(check_digit)

    # Caso 11 caracteres → validar o corregir
    if len(container_code) == 11:
        last_digit = container_code[10]
        if last_digit.isdigit() and int(last_digit) == check_digit:
            return container_code  # es válido
        else:
            # Corregir último carácter
            return code_10 + str(check_digit)

    # Caso más de 11 caracteres → recortar y recalcular
    if len(container_code) > 11:
        return code_10 + str(check_digit)

    return None



########################################################################################################################



def predict(image):
    detecciones_yolo = {}
    detecciones_easy = {}
    crops_con_labels = []
    texto_reconstruido_imgs = []

    # Variables auxiliares para armar code-container
    cn11_code_yolo, cn4_code_yolo, cn7_code_yolo = None, None, None
    cn11_code_easy, cn4_code_easy, cn7_code_easy = None, None, None

    # 1. Detección con primer modelo (IDs)
    results_id = ids_model.predict(image, conf=0.5)

    # 1a. Recolectar todas las detecciones con sus confidences
    detections = []
    for box in results_id[0].boxes:
        cls_id = int(box.cls[0].item())
        cls_name = ids_model.names[cls_id]
        conf = float(box.conf[0].item())
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        detections.append({
            "cls_name": cls_name,
            "conf": conf,
            "coords": (x1, y1, x2, y2)
        })

    # 1b. Filtrar solo la detección de mayor confidence por clase
    best_detections = {}
    for det in detections:
        cls_name = det["cls_name"]
        if cls_name not in best_detections or det["conf"] > best_detections[cls_name]["conf"]:
            best_detections[cls_name] = det

    # Imagen con todas las bounding boxes originales
    img_with_boxes = results_id[0].plot()
    img_with_boxes_pil = Image.fromarray(img_with_boxes)

    # 2. Procesar cada detección filtrada
    for cls_name, det in best_detections.items():
        x1, y1, x2, y2 = det["coords"]
        crop = image.crop((x1, y1, x2, y2))

        # 2a. Pasar crop al modelo OCR (YOLO chars)
        results_char = char_model.predict(crop, conf=0.5)
        chars_detected = []

        for cbox in results_char[0].boxes:
            c_cls_id = int(cbox.cls[0].item())
            c_cls_name = char_model.names[c_cls_id]
            cx1, cy1, cx2, cy2 = cbox.xyxy[0].tolist()
            char_crop = crop.crop((cx1, cy1, cx2, cy2))
            chars_detected.append((cx1, cy1, c_cls_name, char_crop))

        # 2b. Ordenar y concatenar caracteres para YOLO char
        text_pred = ""
        if cls_name in ["cn-11", "iso-type"]:
            if crop.height > crop.width * 1.5:  # vertical
                chars_detected = sorted(chars_detected, key=lambda x: x[1])
            else:  # horizontal
                chars_detected = sorted(chars_detected, key=lambda x: x[0])
            text_pred = "".join([c[2] for c in chars_detected])
        elif cls_name in ["cn-4", "cn-7"]:
            chars_detected = sorted(chars_detected, key=lambda x: x[0])
            text_pred = "".join([c[2] for c in chars_detected])

        # Guardar detecciones YOLO
        detecciones_yolo[cls_name] = text_pred
        if cls_name == "cn-11":
            cn11_code_yolo = text_pred
        elif cls_name == "cn-4":
            cn4_code_yolo = text_pred
        elif cls_name == "cn-7":
            cn7_code_yolo = text_pred

        # 2c. Guardar crops anotados
        crop_with_boxes = results_char[0].plot()
        crops_con_labels.append(Image.fromarray(crop_with_boxes))

        # 2d. EasyOCR
        # Regla: cn-4 / cn-7 horizontales → OCR siempre sobre crop original
        if cls_name in ["cn-4", "cn-7", "cn-11", "iso-type"] and crop.width > crop.height:
            ocr_text = ocr_model.readtext(np.array(crop), detail=0)
        else:
            if chars_detected:
                # Reconstrucción horizontal de chars
                widths, heights = zip(*(c[3].size for c in chars_detected))
                total_width = sum(widths)
                max_height = max(heights)
                new_img = Image.new("RGB", (total_width, max_height), color=(0,0,0))
                x_offset = 0
                for _, _, _, char_crop in chars_detected:
                    new_img.paste(char_crop, (x_offset,0))
                    x_offset += char_crop.width
                texto_reconstruido_imgs.append(new_img)

                ocr_text = ocr_model.readtext(np.array(new_img), detail=0)
            else:
                # No hay chars detectados → OCR sobre crop original
                ocr_text = ocr_model.readtext(np.array(crop), detail=0)

        # Limpiar espacios y guardar en detecciones_easy
        if ocr_text:
            ocr_text_clean = "".join(ocr_text).replace(" ", "")
            ocr_text_clean = re.sub(r'[^A-Z0-9]', '', ocr_text_clean.upper())
            detecciones_easy[cls_name] = ocr_text_clean
            if cls_name == "cn-11":
                cn11_code_easy = ocr_text_clean
            elif cls_name == "cn-4":
                cn4_code_easy = ocr_text_clean
            elif cls_name == "cn-7":
                cn7_code_easy = ocr_text_clean

    # 3. Construir code-container YOLO
    if cn11_code_yolo:
        detecciones_yolo["code-container"] = cn11_code_yolo
    elif cn4_code_yolo and cn7_code_yolo:
        detecciones_yolo["code-container"] = cn4_code_yolo + cn7_code_yolo

    # 4. Construir code-container EasyOCR
    if cn11_code_easy:
        detecciones_easy["code-container"] = cn11_code_easy
    elif cn4_code_easy and cn7_code_easy:
        detecciones_easy["code-container"] = cn4_code_easy + cn7_code_easy

    # 5. Validar ambos
    parsed_yolo = parse_detecciones(detecciones_yolo, rules)
    parsed_easy = parse_detecciones(detecciones_easy, rules) if detecciones_easy else {}

    # 6. Calcular validated_code_container para ambos
    validated_yolo = None
    validated_easy = None

    if "code-container" in detecciones_yolo:
        validated_yolo = calculate_check_digit(detecciones_yolo["code-container"])

    if "code-container" in detecciones_easy:
        validated_easy = calculate_check_digit(detecciones_easy["code-container"])

    # 7. Armar salida final
    salida_json = {
        "output_yolo_char": detecciones_yolo,
        "output_easy_ocr": detecciones_easy,
        "validation": {
            "yolo_char": parsed_yolo,
            "easy_ocr": parsed_easy
        },
        "validated_code_container": {
            "yolo_char": validated_yolo,
            "easy_ocr": validated_easy
        }
    }

    return img_with_boxes_pil, crops_con_labels, texto_reconstruido_imgs, salida_json

def predict_frame(frame_bgr):
    """
    Recibe un frame en formato BGR (OpenCV).
    Devuelve el frame anotado y el JSON con detecciones.
    """
    image_pil = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    img_with_boxes_pil, _, _, salida_json = predict(image_pil)

    # Convertir frame anotado a OpenCV
    annotated_frame = cv2.cvtColor(np.array(img_with_boxes_pil), cv2.COLOR_RGB2BGR)

    # ======================
    # Mostrar solo códigos validados
    # ======================
    validated = salida_json.get("validated_code_container", {})
    detected_code = None

    if validated.get("yolo_char"):
        detected_code = validated["yolo_char"]
    elif validated.get("easy_ocr"):
        detected_code = validated["easy_ocr"]

    if detected_code:
        cv2.putText(
            annotated_frame,
            f"Detected: {detected_code}",
            (30, 50),  # posición en pantalla
            cv2.FONT_HERSHEY_SIMPLEX,
            1.2,
            (0, 255, 0),  # verde = válido
            3,
            cv2.LINE_AA
        )

    return annotated_frame, salida_json


# ======================
# Procesar video
# ======================
video_path = "/home/gnz/GitHub/yolo11_container/videos/short_youtube.mp4"
cap = cv2.VideoCapture(video_path)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(
    "output_with_codes.mp4",
    fourcc,
    cap.get(cv2.CAP_PROP_FPS),
    (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
)

frame_id = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    annotated_frame, salida_json = predict_frame(frame)

    # Mostrar
    cv2.imshow("Container OCR Video", annotated_frame)

    # Guardar
    out.write(annotated_frame)

    # Log cada 30 frames
    if frame_id % 30 == 0:
        print(f"Frame {frame_id}: {salida_json}")

    frame_id += 1

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
out.release()
cv2.destroyAllWindows()
   



Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.



0: 640x384 (no detections), 122.9ms
Speed: 2.4ms preprocess, 122.9ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)
Frame 0: {'output_yolo_char': {}, 'output_easy_ocr': {}, 'validation': {'yolo_char': {}, 'easy_ocr': {}}, 'validated_code_container': {'yolo_char': None, 'easy_ocr': None}}

0: 640x384 (no detections), 105.9ms
Speed: 2.1ms preprocess, 105.9ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 118.9ms
Speed: 2.3ms preprocess, 118.9ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 128.8ms
Speed: 5.1ms preprocess, 128.8ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 94.7ms
Speed: 1.9ms preprocess, 94.7ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 101.9ms
Speed: 2.4ms preprocess, 101.9ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no det

/home/gnz/GitHub/yolo11_container/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


0: 640x384 1 cn-11, 98.9ms
Speed: 4.3ms preprocess, 98.9ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 384)

0: 640x64 2 0s, 1 1, 1 2, 2 5s, 2 7s, 1 R, 29.7ms
Speed: 1.1ms preprocess, 29.7ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 64)

0: 640x384 1 cn-11, 94.4ms
Speed: 2.5ms preprocess, 94.4ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x64 2 0s, 1 1, 1 2, 2 5s, 2 7s, 1 R, 1 U, 26.2ms
Speed: 0.7ms preprocess, 26.2ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 64)

0: 640x384 (no detections), 133.3ms
Speed: 1.9ms preprocess, 133.3ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 128.5ms
Speed: 2.4ms preprocess, 128.5ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)
Frame 120: {'output_yolo_char': {}, 'output_easy_ocr': {}, 'validation': {'yolo_char': {}, 'easy_ocr': {}}, 'validated_code_container': {'yolo_char': None, 'easy_ocr': None}}

0: 640

In [ ]:
# GUARDAS LOS CROPS DE CADA FRAME DETECTADO
import os


# Cargar modelo
model = YOLO("/home/gnz/GitHub/yolo11_container/YOLO_IDs/ID_YOLO_container/weights/best.pt")

# Crear carpeta para guardar crops
output_dir = "detections_crops"
os.makedirs(output_dir, exist_ok=True)

# Video de entrada
video_path = "/home/gnz/GitHub/yolo11_container/videos/short_youtube.mp4" 
cap = cv2.VideoCapture(video_path)

frame_id = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Inferencia en el frame
    results = model(frame, conf=0.5)  # puedes ajustar el umbral aquí

    for r in results:
        for box in r.boxes:
            # Coordenadas de la caja (x1, y1, x2, y2)
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Confianza
            conf = float(box.conf[0])

            # Clase detectada
            cls = int(box.cls[0])
            label = model.names[cls]

            if conf >= 0.5:  # aplica el umbral de confianza
                # Crop
                crop = frame[y1:y2, x1:x2]

                # Nombre del archivo
                crop_filename = f"{output_dir}/frame{frame_id}_{label}_{conf:.2f}.jpg"

                # Guardar crop
                cv2.imwrite(crop_filename, crop)

                # Dibujar bbox en el frame (opcional)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, f"{label} {conf:.2f}", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Mostrar en pantalla
    cv2.imshow("YOLO detections", frame)
    frame_id += 1

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()



0: 640x384 (no detections), 121.2ms
Speed: 30.5ms preprocess, 121.2ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 223.6ms
Speed: 8.0ms preprocess, 223.6ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 94.3ms
Speed: 2.2ms preprocess, 94.3ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 89.8ms
Speed: 6.7ms preprocess, 89.8ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 107.7ms
Speed: 1.9ms preprocess, 107.7ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 89.1ms
Speed: 2.2ms preprocess, 89.1ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 91.5ms
Speed: 1.9ms preprocess, 91.5ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 (no detections), 108.4ms
Speed: 1.9ms preprocess, 

In [2]:
from ultralytics import YOLO
import cv2

# Modelos
model_main = YOLO("/home/gnz/GitHub/yolo11_container/YOLO_IDs/ID_YOLO_container/weights/best.pt")        # Modelo principal (ej: detectar placas)
model_chars = YOLO("/home/gnz/GitHub/yolo11_container/YOLO_Characters/Character_YOLO_container_finetune_extra_large_phase2/weights/best.pt")      # Modelo para caracteres

# Video
cap = cv2.VideoCapture("/home/gnz/GitHub/yolo11_container/videos/short_youtube")  # o "rtsp://user:pass@ip:puerto/stream"

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # --------------------
    # Primera detección (objetos grandes)
    # --------------------
    results_main = model_main(frame, conf=0.5)

    for r in results_main:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            cls = int(box.cls[0])
            label = model_main.names[cls]

            if conf >= 0.5:
                # Crop del objeto detectado
                crop = frame[y1:y2, x1:x2]

                # --------------------
                # Segunda detección (caracteres dentro del crop)
                # --------------------
                results_chars = model_chars(crop, conf=0.4)

                for rc in results_chars:
                    for cbox in rc.boxes:
                        cx1, cy1, cx2, cy2 = map(int, cbox.xyxy[0])
                        cconf = float(cbox.conf[0])
                        ccls = int(cbox.cls[0])
                        clabel = model_chars.names[ccls]

                        if cconf >= 0.4:
                            # Dibujar detecciones dentro del crop
                            cv2.rectangle(crop, (cx1, cy1), (cx2, cy2), (0, 0, 255), 2)
                            cv2.putText(crop, f"{clabel} {cconf:.2f}", (cx1, cy1 - 5),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

                # Reemplazar crop anotado en el frame original
                frame[y1:y2, x1:x2] = crop

                # Dibujar bbox del objeto grande
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, f"{label} {conf:.2f}", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Mostrar en pantalla
    cv2.imshow("YOLO Cascade Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
